In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Step 1: Create the original dataframe
df = pd.read_excel("test data v3 120525.xlsx")

df.sample()

In [ ]:
df.columns

In [ ]:
# df["Opportunity Value"] = (df["Total Contract Value"] + df["Strategic Value "] + df["Customer Intimacy"]) / 3
# df["Delivery Confidence"] =(df["GDIT  Capability "] + df["Solution Complexity"] + 5 - df["Implementation Risk"]) / 3
# df = df.round(2)
# df

In [ ]:
# Step 2: Consolidate projects with same Metric A & Metric B
df_grouped = (
    df.groupby(["Opportunity Value ", "Delivery  Confidence "])["PID"]
      .apply(lambda names: "&".join(names))  # concatenate names
      .reset_index()
)

df_grouped

In [ ]:

def quadrant(df):

    # Step 3: Create Magic Quadrant chart
    fig = px.scatter(
        df,
        x="Delivery  Confidence ",
        y="Opportunity Value ",
        text="PID",
        width=800,
        height=800
    )

    # Add quadrant lines at midpoint (3)
    fig.add_shape(type="line", x0=3, y0=1, x1=3, y1=5, line=dict(color="gray", dash="dash"))
    fig.add_shape(type="line", x0=1, y0=3, x1=5, y1=3, line=dict(color="gray", dash="dash"))

    fig.add_shape(type="line", x0=1, y0=1, x1=1, y1=5, line=dict(color="black"))
    fig.add_shape(type="line", x0=1, y0=1, x1=5, y1=1, line=dict(color="black"))
    fig.add_shape(type="line", x0=5, y0=1, x1=5, y1=5, line=dict(color="black"))
    fig.add_shape(type="line", x0=1, y0=5, x1=5, y1=5, line=dict(color="black"))

    fig.add_annotation(x=4, y=5, xanchor="center", yanchor="top", text="Pursue", showarrow=False, font=dict(family="Helvetica Bold",size=18, color="blue"))
    fig.add_annotation(x=2, y=5, xanchor="center", yanchor="top",  text="Sub/Partnership", showarrow=False, font=dict(family="Helvetica Bold",size=18, color="blue"))
    fig.add_annotation(x=2, y=1, xanchor="center", yanchor="bottom", text="Sunset", showarrow=False, font=dict(family="Helvetica Bold",size=18, color="blue"))
    fig.add_annotation(x=4, y=1, xanchor="center", yanchor="bottom", text="Low ROI", showarrow=False, font=dict(family="Helvetica Bold",size=18, color="blue"))

    # Adjust text labels
    fig.update_traces(textposition="top center")

 
    return fig

magic_q = quadrant(df_grouped)

magic_q.show()


In [ ]:

fig = make_subplots(
    rows=1, cols=2,
    horizontal_spacing=0.025,
    column_widths=[0.75, 0.25],
    specs=[[{"type": "xy"}, {"type": "table"}]],
    subplot_titles=(None, None)
)

# Left subplot (scatter)
for trace in magic_q.data:
    fig.add_trace(trace, row=1, col=1)

# Copy annotations
if hasattr(magic_q.layout, "annotations"):
    fig.update_layout(annotations=magic_q.layout.annotations)

# Copy shapes
if hasattr(magic_q.layout, "shapes"):
    fig.update_layout(shapes=magic_q.layout.shapes)


# Right subplot (table)
table = go.Table(
    columnwidth=[1,4],
    header=dict(values=["PID","Project"], fill_color="lightgray", align="left", font=dict(size=12)),
    cells=dict(values=[df["PID"], df["Project"]],  align="left", height=22.5, font=dict(size=12))
)

fig.add_trace(table, row=1, col=2)

fig.update_layout(
    xaxis=dict(dtick=1, title="Delivery Confidence"),
    yaxis=dict(dtick=1, title="Opportunity Value"),
    width=1200, 
    height=800,
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(size=16, color="black", family="Helvetica Bold")
)


fig.show()


In [ ]:
df.to_excel("Output.xlsx", index=False)